# 3.3 Real measurements on healthy volunteers

In [ ]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from scipy import signal
from IPython.display import display

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from style import (
    OUTPUT_DIR,
    PRIMARY_COLOR,
    SECONDARY_COLOR,
    INTERVAL_FILL,
    EVENT_FILL,
    GRID_COLOR,
    DARK_TEXT,
    finish_axis,
    save_figure,
)

In [ ]:
VOLUNTEER_ROOT = Path.home() / "TFM QMUL" / "third trim" / "Volunteers"
ACC_DIR = VOLUNTEER_ROOT / "acc"
MIC_DIR = VOLUNTEER_ROOT / "mic"
VOLUNTEERS_ACC_DIR = ACC_DIR
VOLUNTEERS_MIC_DIR = MIC_DIR

DVC_REPO_ROOT = Path.home() / "Desktop" / "dvc"
NPZ_DIR = DVC_REPO_ROOT / "pipeline" / "wav_to_npz"

APPENDIX_CANDIDATES = [
    DVC_REPO_ROOT / "Online_Appendix_training_set.csv",
    Path.home()
    / "Desktop"
    / "QMUL BME"
    / "TFM"
    / "PHYSIONET DATA"
    / "Online_Appendix_training_set.csv",
]


def first_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("PhysioNet appendix file not found.")


APPENDIX_TRAINING_SET = first_existing_path(APPENDIX_CANDIDATES)

### 2.5.1 Experimental protocol — recording manifest

In [ ]:
def condition_from_name(filename):
    lower = filename.lower()
    if "normalbreathing" in lower:
        return "normal"
    if "holdbreathing" in lower or "holdingbreath" in lower:
        return "hold"
    raise ValueError(f"Unknown breathing condition: {filename}")


def subject_from_name(filename):
    match = re.search(r"subj(\d+)", filename.lower())
    if not match:
        raise ValueError(f"Subject ID not found: {filename}")
    return int(match.group(1))


records = []

for sensor_type, folder in [("ACC", VOLUNTEERS_ACC_DIR), ("MIC", VOLUNTEERS_MIC_DIR)]:
    for path in sorted(folder.glob("*.txt")):
        records.append(
            {
                "filepath": path,
                "filename": path.name,
                "sensor_type": sensor_type,
                "subject_id": subject_from_name(path.name),
                "condition": condition_from_name(path.name),
            }
        )

volunteer_manifest = (
    pd.DataFrame(records)
    .sort_values(["subject_id", "sensor_type", "condition"])
    .reset_index(drop=True)
)

if len(volunteer_manifest) != 24:
    display(volunteer_manifest)
    raise ValueError(f"Expected 24 final recordings; found {len(volunteer_manifest)}.")

display(volunteer_manifest)

### 3.3.1 Autocorrelation-derived rate estimates

In [ ]:
LOW_PASS_HZ = 400.0
HIGH_PASS_HZ = 25.0
FILTER_ORDER = 2
ENVELOPE_LOW_PASS_HZ = 8.0
MIN_LAG_S = 0.5
MAX_LAG_S = 2.0
SPIKE_WINDOW_S = 0.5
SPIKE_THRESHOLD_MULTIPLIER = 3.0

PHYSIONET_RANDOM_SEED = 42
N_PHYSIONET_SUBJECTS = 6

In [ ]:
SUBJECT_PATTERN = re.compile(r"subj(\d+)", re.IGNORECASE)


def parse_volunteer_filename(filepath):
    name = filepath.stem
    lower_name = name.lower()
    upper_name = name.upper()

    if upper_name.startswith("ACC"):
        sensor_type = "ACC"
    elif upper_name.startswith("MIC"):
        sensor_type = "MIC"
    else:
        return None

    subject_match = SUBJECT_PATTERN.search(name)
    subject_id = f"subj{subject_match.group(1)}" if subject_match else None

    if "normal" in lower_name:
        condition = "normalbreathing"
    elif "hold" in lower_name:
        condition = "holdbreathing"
    else:
        condition = None

    if not all((subject_id, condition)):
        return None

    return {
        "filepath": filepath,
        "sensor_type": sensor_type,
        "subject_id": subject_id,
        "condition": condition,
        "recording_id": f"{sensor_type}_{subject_id}_{condition}",
    }


def load_labchart_mean_channel(filepath):

    frame = pd.read_csv(filepath, sep="\t", header=None, decimal=",", engine="python").apply(
        pd.to_numeric, errors="coerce"
    )

    if frame.shape[1] < 10:
        raise ValueError(
            f"{filepath.name}: expected at least 10 original columns, "
            f"found {frame.shape[1]}."
        )

    time_values = frame.iloc[:, 0].to_numpy(dtype=float)
    individual_channels = frame.iloc[:, 1:9].copy()
    individual_channels.columns = [f"ch{i}" for i in range(1, 9)]
    mean_signal = frame.iloc[:, 9].to_numpy(dtype=float)

    valid_time = np.isfinite(time_values)
    time_values = time_values[valid_time]
    individual_channels = individual_channels.loc[valid_time].reset_index(drop=True)
    mean_signal = mean_signal[valid_time]

    mean_signal = (
        pd.Series(mean_signal)
        .replace([np.inf, -np.inf], np.nan)
        .interpolate(method="linear", limit_direction="both")
        .to_numpy(dtype=float)
    )

    if len(time_values) < 16 or not np.isfinite(mean_signal).all():
        raise ValueError(f"{filepath.name}: unusable time or mean-channel data.")

    time_steps = np.diff(time_values)
    time_steps = time_steps[np.isfinite(time_steps) & (time_steps > 0)]
    if len(time_steps) == 0:
        raise ValueError(f"{filepath.name}: sampling frequency could not be estimated.")

    fs = float(1.0 / np.median(time_steps))
    n_individual_channels = int(
        sum(
            np.isfinite(individual_channels[column].to_numpy(dtype=float)).any()
            for column in individual_channels.columns
        )
    )

    return time_values, mean_signal, fs, n_individual_channels


def zero_phase_filter(values, fs, cutoff_hz, btype, order):
    b, a = signal.butter(order, cutoff_hz, btype=btype, fs=fs)
    return signal.filtfilt(b, a, values)


def schmidt_spike_removal(values, fs, maximum_iterations=10000):
    values = np.asarray(values, dtype=float).copy()
    window_size = int(round(SPIKE_WINDOW_S * fs))
    trailing_samples = len(values) % window_size

    if trailing_samples:
        main_signal = values[:-trailing_samples]
        trailing_signal = values[-trailing_samples:]
    else:
        main_signal = values
        trailing_signal = np.array([], dtype=float)

    if len(main_signal) == 0:
        return values

    windows = main_signal.reshape(-1, window_size)

    for _ in range(maximum_iterations):
        maxima = np.max(np.abs(windows), axis=1)
        median_maximum = float(np.median(maxima))

        if not np.isfinite(median_maximum) or median_maximum <= 0:
            break

        if not np.any(maxima > SPIKE_THRESHOLD_MULTIPLIER * median_maximum):
            break

        window_index = int(np.argmax(maxima))
        selected_window = windows[window_index]
        spike_position = int(np.argmax(np.abs(selected_window)))

        zero_crossings = np.concatenate(
            [np.abs(np.diff(np.sign(selected_window))) > 1, np.array([False])]
        )

        previous = np.flatnonzero(zero_crossings[: spike_position + 1])
        following = np.flatnonzero(zero_crossings[spike_position + 1 :])

        spike_start = int(previous[-1]) if len(previous) else 0
        spike_end = (
            int(spike_position + 1 + following[0]) if len(following) else window_size - 1
        )

        windows[window_index, spike_start : spike_end + 1] = 0.0001
    else:
        raise RuntimeError("Schmidt spike removal reached its safety iteration limit.")

    return np.concatenate([windows.reshape(-1), trailing_signal])


def homomorphic_envelope(values, fs):
    analytic_magnitude = np.abs(signal.hilbert(values))
    analytic_magnitude = np.maximum(analytic_magnitude, np.finfo(float).tiny)
    log_magnitude = np.log(analytic_magnitude)

    b, a = signal.butter(1, ENVELOPE_LOW_PASS_HZ, btype="lowpass", fs=fs)

    envelope = np.exp(signal.filtfilt(b, a, log_magnitude))
    if len(envelope) > 1:
        envelope[0] = envelope[1]
    return envelope


def estimate_heart_rate_schmidt(values, fs):
    values = np.asarray(values, dtype=float).reshape(-1)

    if fs <= 2 * LOW_PASS_HZ:
        raise ValueError(f"Sampling frequency must be greater than {2 * LOW_PASS_HZ:.0f} Hz.")

    low_passed = zero_phase_filter(values, fs, LOW_PASS_HZ, "lowpass", FILTER_ORDER)
    band_limited = zero_phase_filter(low_passed, fs, HIGH_PASS_HZ, "highpass", FILTER_ORDER)
    filtered_signal = schmidt_spike_removal(band_limited, fs)
    envelope = homomorphic_envelope(filtered_signal, fs)

    centred_envelope = envelope - np.mean(envelope)
    zero_lag_energy = float(np.dot(centred_envelope, centred_envelope))

    if zero_lag_energy <= 0 or not np.isfinite(zero_lag_energy):
        raise ValueError("Autocorrelation could not be normalised.")

    full_autocorrelation = signal.correlate(
        centred_envelope, centred_envelope, mode="full", method="fft"
    )

    n_samples = len(centred_envelope)
    autocorrelation = full_autocorrelation[n_samples:] / zero_lag_energy
    lag_seconds = np.arange(1, len(autocorrelation) + 1, dtype=float) / fs

    search_indices = np.flatnonzero((lag_seconds >= MIN_LAG_S) & (lag_seconds <= MAX_LAG_S))

    if len(search_indices) == 0:
        raise ValueError("No lags were available inside the 0.5-2.0 s interval.")

    selected_index = int(search_indices[np.argmax(autocorrelation[search_indices])])
    selected_lag_s = float(lag_seconds[selected_index])

    return {
        "filtered_signal": filtered_signal,
        "envelope": envelope,
        "lag_seconds": lag_seconds,
        "autocorrelation": autocorrelation,
        "selected_index": selected_index,
        "selected_lag_s": selected_lag_s,
        "estimated_heart_rate_bpm": float(60.0 / selected_lag_s),
        "autocorrelation_peak": float(autocorrelation[selected_index]),
    }


def load_volunteer_recordings():
    metadata_rows = []

    for directory in (ACC_DIR, MIC_DIR):
        for filepath in sorted(directory.glob("*.txt")):
            parsed = parse_volunteer_filename(filepath)
            if parsed is not None:
                metadata_rows.append(parsed)

    recordings = {}
    failures = []

    for metadata in metadata_rows:
        try:
            time_values, mean_signal, fs, n_channels = load_labchart_mean_channel(
                metadata["filepath"]
            )
            result = estimate_heart_rate_schmidt(mean_signal, fs)

            recordings[metadata["recording_id"]] = {
                **metadata,
                "time": time_values,
                "mean_signal": mean_signal,
                "fs_hz": fs,
                "n_individual_channels_available": n_channels,
                "result": result,
            }
        except Exception as error:
            failures.append((str(metadata["filepath"]), str(error)))

    print(f"Loaded recordings: {len(recordings)}")
    print(f"Load failures: {len(failures)}")

    for filepath, message in failures:
        print(f"FAILED: {filepath}\\n  {message}")

    if failures or len(recordings) != 24:
        raise RuntimeError(
            f"Expected 24 successful volunteer recordings; obtained {len(recordings)}."
        )

    return recordings


recordings = load_volunteer_recordings()

#### Table 3 — Autocorrelation-derived rate estimates obtained from the ACC and MIC recordings

In [ ]:
rows = []

for recording_id, recording in recordings.items():
    result = recording["result"]
    rows.append(
        {
            "recording_id": recording_id,
            "subject_id": recording["subject_id"],
            "sensor_type": recording["sensor_type"],
            "condition": recording["condition"],
            "estimated_heart_rate_bpm": result["estimated_heart_rate_bpm"],
            "selected_cycle_period_s": result["selected_lag_s"],
            "autocorrelation_peak": result["autocorrelation_peak"],
        }
    )

recording_results = (
    pd.DataFrame(rows)
    .sort_values(["condition", "subject_id", "sensor_type"])
    .reset_index(drop=True)
)

recording_results.to_csv(
    OUTPUT_DIR / "recording_level_autocorrelation_results.csv", index=False
)

group_summary = (
    recording_results.groupby(["sensor_type", "condition"])["estimated_heart_rate_bpm"]
    .agg(n="count", median_bpm="median", minimum_bpm="min", maximum_bpm="max")
    .reset_index()
)

group_summary.to_csv(OUTPUT_DIR / "group_estimated_heart_rate_summary.csv", index=False)

print("Group summary")
display(
    group_summary.style.format(
        {"median_bpm": "{:.1f}", "minimum_bpm": "{:.1f}", "maximum_bpm": "{:.1f}"}
    )
)

### 3.3.2 Agreement between sensing modalities

#### Figure 16 — Relationship between derived rate estimates obtained from the accelerometer and microphone

In [ ]:
def paired_values(condition):

    subset = recording_results.loc[recording_results["condition"] == condition]

    paired = subset.pivot(
        index="subject_id", columns="sensor_type", values="estimated_heart_rate_bpm"
    ).sort_index()

    required_columns = {"ACC", "MIC"}

    if not required_columns.issubset(paired.columns):
        missing = required_columns.difference(paired.columns)
        raise RuntimeError(f"Missing sensor columns for {condition}: {sorted(missing)}")

    paired = paired[["ACC", "MIC"]].dropna()

    if len(paired) < 2:
        raise RuntimeError(f"Fewer than two complete ACC-MIC pairs for {condition}.")

    return paired


conditions = [
    ("normalbreathing", "(a) Normal breathing"),
    ("holdbreathing", "(b) Breath hold"),
]

paired_by_condition = {condition: paired_values(condition) for condition, _ in conditions}

all_estimates = np.concatenate(
    [
        paired[["ACC", "MIC"]].to_numpy(dtype=float).ravel()
        for paired in paired_by_condition.values()
    ]
)

plot_min = 5.0 * np.floor((np.nanmin(all_estimates) - 2.0) / 5.0)

plot_max = 5.0 * np.ceil((np.nanmax(all_estimates) + 2.0) / 5.0)

fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)

regression_rows = []

for ax, (condition, title) in zip(axes, conditions):
    paired = paired_by_condition[condition]

    acc = paired["ACC"].to_numpy(dtype=float)
    mic = paired["MIC"].to_numpy(dtype=float)

    slope, intercept = np.polyfit(acc, mic, 1)

    correlation = float(np.corrcoef(acc, mic)[0, 1])

    regression_rows.append(
        {
            "condition": condition,
            "n_pairs": len(paired),
            "slope": slope,
            "intercept": intercept,
            "pearson_r": correlation,
            "r_squared": correlation**2,
        }
    )

    ax.scatter(
        acc,
        mic,
        s=52,
        color=PRIMARY_COLOR,
        edgecolors="white",
        linewidths=0.6,
        zorder=3,
        label="Participants",
    )

    ax.plot(
        [plot_min, plot_max],
        [plot_min, plot_max],
        linestyle="--",
        linewidth=1.2,
        color=SECONDARY_COLOR,
        alpha=0.80,
        label="Line of identity",
        zorder=1,
    )

    fit_x = np.linspace(float(np.min(acc)), float(np.max(acc)), 200)

    fit_y = slope * fit_x + intercept

    ax.plot(fit_x, fit_y, color=PRIMARY_COLOR, linewidth=1.8, label="Best-fit line", zorder=2)

    ax.set_xlim(plot_min, plot_max)

    ax.set_ylim(plot_min, plot_max)

    ax.set_xlabel("Accelerometer estimate (bpm)")

    ax.set_title(title, loc="left", fontweight="semibold")

    finish_axis(ax, grid_axis="both")

axes[0].set_ylabel("Microphone estimate (bpm)")

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False
)

fig.tight_layout(rect=[0, 0.08, 1, 1])

save_figure(fig, "estimated_heart_rate_agreement_scatter")

plt.show()

regression_summary = pd.DataFrame(regression_rows)

regression_summary.to_csv(
    OUTPUT_DIR / "estimated_heart_rate_regression_summary.csv", index=False
)

print("ACC-microphone regression summary")

display(
    regression_summary.style.format(
        {
            "slope": "{:.3f}",
            "intercept": "{:.2f}",
            "pearson_r": "{:.3f}",
            "r_squared": "{:.3f}",
        }
    )
)

#### Figure 17 — Bland–Altman analysis of accelerometer- and microphone-derived rate estimates

In [ ]:
bland_altman_summary_rows = []
bland_altman_participant_rows = []

fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)

for ax, (condition, title) in zip(axes, conditions):
    paired = paired_by_condition[condition]

    acc = paired["ACC"].to_numpy(dtype=float)
    mic = paired["MIC"].to_numpy(dtype=float)

    means = (acc + mic) / 2.0

    differences = mic - acc

    bias = float(np.mean(differences))

    difference_sd = float(np.std(differences, ddof=1))

    lower_loa = bias - 1.96 * difference_sd

    upper_loa = bias + 1.96 * difference_sd

    bland_altman_summary_rows.append(
        {
            "condition": condition,
            "n_pairs": len(paired),
            "mean_difference_bpm": bias,
            "sd_difference_bpm": difference_sd,
            "lower_95_loa_bpm": lower_loa,
            "upper_95_loa_bpm": upper_loa,
        }
    )

    for subject_id, mean_value, difference in zip(paired.index, means, differences):
        bland_altman_participant_rows.append(
            {
                "subject_id": subject_id,
                "condition": condition,
                "mean_acc_mic_bpm": mean_value,
                "mic_minus_acc_bpm": difference,
            }
        )

    ax.scatter(
        means,
        differences,
        s=52,
        color=PRIMARY_COLOR,
        edgecolors=PRIMARY_COLOR,
        linewidths=0.6,
        zorder=3,
        label="Participants",
    )

    ax.axhline(0, color=GRID_COLOR, linewidth=1.0, zorder=0)

    ax.axhline(bias, color=PRIMARY_COLOR, linewidth=1.8, label="Mean difference", zorder=2)

    ax.axhline(
        lower_loa,
        color=SECONDARY_COLOR,
        linestyle="--",
        linewidth=1.2,
        label="95% limits of agreement",
        zorder=1,
    )

    ax.axhline(upper_loa, color=SECONDARY_COLOR, linestyle="--", linewidth=1.2, zorder=1)

    ax.set_xlabel("Mean of accelerometer and microphone estimates (bpm)")

    ax.set_title(title, loc="left", fontweight="semibold")

    finish_axis(ax, grid_axis="both")

axes[0].set_ylabel("Microphone − accelerometer (bpm)")

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.03), ncol=3, frameon=False
)

fig.tight_layout(rect=[0, 0.09, 1, 1])

save_figure(fig, "estimated_heart_rate_bland_altman")

plt.show()

bland_altman_summary = pd.DataFrame(bland_altman_summary_rows)

bland_altman_participant_data = pd.DataFrame(bland_altman_participant_rows)

bland_altman_summary.to_csv(OUTPUT_DIR / "bland_altman_summary.csv", index=False)

bland_altman_participant_data.to_csv(
    OUTPUT_DIR / "bland_altman_participant_data.csv", index=False
)

print("Bland-Altman summary")

display(
    bland_altman_summary.style.format(
        {
            "mean_difference_bpm": "{:.2f}",
            "sd_difference_bpm": "{:.2f}",
            "lower_95_loa_bpm": "{:.2f}",
            "upper_95_loa_bpm": "{:.2f}",
        }
    )
)

### 3.3.3 Waveform comparison

In [ ]:
def load_physionet_candidates():
    appendix = pd.read_csv(APPENDIX_TRAINING_SET)
    appendix["Diagnosis"] = appendix["Diagnosis"].astype(str).str.strip()
    appendix["Database"] = appendix["Database"].astype(str).str.strip()

    normal_b = appendix.loc[
        (appendix["Database"] == "training-b")
        & (appendix["Diagnosis"] == "Normal")
        & appendix["Subject ID"].notna(),
        ["Challenge record name", "Subject ID"],
    ].drop_duplicates()

    available_subjects = []
    for subject_id, group in normal_b.groupby("Subject ID"):
        if any(
            (NPZ_DIR / f"{record_name}.npz").exists()
            for record_name in group["Challenge record name"]
        ):
            available_subjects.append(subject_id)

    if len(available_subjects) < N_PHYSIONET_SUBJECTS:
        raise RuntimeError(
            "Fewer than six PhysioNet Normal subjects have available NPZ files."
        )

    rng = np.random.default_rng(PHYSIONET_RANDOM_SEED)
    chosen_subjects = rng.choice(
        np.asarray(available_subjects, dtype=object), size=N_PHYSIONET_SUBJECTS, replace=False
    )

    selected_rows = normal_b.loc[normal_b["Subject ID"].isin(chosen_subjects)]

    candidates = []

    for _, row in selected_rows.iterrows():
        record_name = row["Challenge record name"]
        npz_path = NPZ_DIR / f"{record_name}.npz"

        if not npz_path.exists():
            continue

        with np.load(npz_path) as data:
            values = np.asarray(data["y"], dtype=float).squeeze()
            fs = float(np.asarray(data["fs"]).squeeze())

        values = np.asarray(values, dtype=float).reshape(-1)
        values = (
            pd.Series(values)
            .replace([np.inf, -np.inf], np.nan)
            .interpolate(method="linear", limit_direction="both")
            .to_numpy(dtype=float)
        )

        if not np.isfinite(values).all():
            continue

        try:
            result = estimate_heart_rate_schmidt(values, fs)
        except Exception:
            continue

        candidates.append(
            {
                "subject_id": row["Subject ID"],
                "record_name": record_name,
                "time": np.arange(len(values), dtype=float) / fs,
                "signal": values,
                "fs_hz": fs,
                "result": result,
            }
        )

    if not candidates:
        raise RuntimeError("No PhysioNet comparison recordings could be processed.")

    return candidates


physionet_candidates = load_physionet_candidates()
print(
    f"Processed {len(physionet_candidates)} PhysioNet recordings "
    f"from {N_PHYSIONET_SUBJECTS} Normal subjects."
)

#### Figure 18 — Representative comparison of recurrent cardiac-related structure (NilocasPatch vs PhysioNet)

In [ ]:
def refine_to_envelope_maximum(envelope, centre, half_width):
    lower = max(0, int(centre - half_width))
    upper = min(len(envelope), int(centre + half_width + 1))
    if upper <= lower:
        return int(np.clip(centre, 0, len(envelope) - 1))
    return lower + int(np.argmax(envelope[lower:upper]))


def find_autocorrelation_anchored_cycles(result, fs):

    envelope = np.asarray(result["envelope"], dtype=float)
    envelope = envelope - np.nanmin(envelope)
    maximum = np.nanmax(envelope)
    if not np.isfinite(maximum) or maximum <= 0:
        raise ValueError("Envelope could not be normalised.")
    envelope = envelope / maximum

    period_samples = int(round(result["selected_lag_s"] * fs))
    refinement_half_width = max(1, int(round(0.12 * period_samples)))

    envelope_peaks, _ = signal.find_peaks(
        envelope, distance=max(1, int(round(0.20 * period_samples)))
    )

    if len(envelope_peaks) == 0:
        raise ValueError("No envelope peaks were found.")

    strongest_candidates = envelope_peaks[
        np.argsort(envelope[envelope_peaks])[-min(80, len(envelope_peaks)) :]
    ]

    best_cycle = None
    best_score = -np.inf

    for anchor in strongest_candidates:
        phase = int(anchor % period_samples)
        expected = np.arange(phase, len(envelope), period_samples, dtype=int)
        primary = np.array(
            [
                refine_to_envelope_maximum(envelope, position, refinement_half_width)
                for position in expected
            ],
            dtype=int,
        )
        primary = np.unique(primary)

        for index in range(len(primary) - 2):
            p1, p2, p3 = map(int, primary[index : index + 3])
            interval_1 = p2 - p1
            interval_2 = p3 - p2

            if not (
                0.75 * period_samples <= interval_1 <= 1.25 * period_samples
                and 0.75 * period_samples <= interval_2 <= 1.25 * period_samples
            ):
                continue

            secondary = []
            for first_primary, next_primary in ((p1, p2), (p2, p3)):
                cycle_length = next_primary - first_primary
                search_start = int(first_primary + 0.15 * cycle_length)
                search_end = int(first_primary + 0.50 * cycle_length)

                if search_end <= search_start:
                    secondary = []
                    break

                secondary.append(
                    search_start + int(np.argmax(envelope[search_start:search_end]))
                )

            if len(secondary) != 2:
                continue

            s2_1, s2_2 = secondary
            regularity_penalty = (
                abs(interval_1 - period_samples) + abs(interval_2 - period_samples)
            ) / period_samples

            score = (
                envelope[p1]
                + envelope[p2]
                + envelope[p3]
                + 0.40 * envelope[s2_1]
                + 0.40 * envelope[s2_2]
                - 0.50 * regularity_penalty
            )

            if score > best_score:
                best_score = score
                best_cycle = {
                    "p1": p1,
                    "p2": p2,
                    "p3": p3,
                    "s2_1": int(s2_1),
                    "s2_2": int(s2_2),
                }

    if best_cycle is None:
        raise ValueError("No two clean autocorrelation-anchored cycles were found.")

    return best_cycle


def required_duration(time_values, cycle, left_margin=0.18, right_margin=0.18):
    start = max(float(time_values[0]), float(time_values[cycle["p1"]]) - left_margin)
    end = min(float(time_values[-1]), float(time_values[cycle["s2_2"]]) + right_margin)
    return end - start


def build_segment(
    time_values,
    filtered_signal,
    cycle,
    target_duration,
    selected_lag_s,
    estimated_rate,
    autocorrelation_peak,
    left_margin=0.18,
    event_half_width=0.035,
):
    p1 = cycle["p1"]
    p2 = cycle["p2"]
    s2_1 = cycle["s2_1"]
    s2_2 = cycle["s2_2"]

    start_time = max(float(time_values[0]), float(time_values[p1]) - left_margin)
    end_time = start_time + target_duration

    if end_time > float(time_values[-1]):
        start_time = max(float(time_values[0]), float(time_values[-1]) - target_duration)
        end_time = start_time + target_duration

    visible = (time_values >= start_time) & (time_values <= end_time)
    x = time_values[visible] - start_time
    y = np.asarray(filtered_signal, dtype=float)[visible].copy()

    scale = np.nanmax(np.abs(y))
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Selected segment could not be normalised.")
    y = y / scale

    event_times = {
        "P1_1": float(time_values[p1]) - start_time,
        "P2_1": float(time_values[s2_1]) - start_time,
        "P1_2": float(time_values[p2]) - start_time,
        "P2_2": float(time_values[s2_2]) - start_time,
    }

    boundaries = {
        "x_start": 0.0,
        "x_end": target_duration,
        "S1_1_start": event_times["P1_1"] - event_half_width,
        "S1_1_end": event_times["P1_1"] + event_half_width,
        "S2_1_start": event_times["P2_1"] - event_half_width,
        "S2_1_end": event_times["P2_1"] + event_half_width,
        "S1_2_start": event_times["P1_2"] - event_half_width,
        "S1_2_end": event_times["P1_2"] + event_half_width,
        "S2_2_start": event_times["P2_2"] - event_half_width,
        "S2_2_end": event_times["P2_2"] + event_half_width,
    }

    return {
        "x": x,
        "y": y,
        "boundaries": boundaries,
        "selected_lag_s": selected_lag_s,
        "estimated_rate": estimated_rate,
        "autocorrelation_peak": autocorrelation_peak,
    }


def plot_segment(ax, segment, title):
    x = segment["x"]
    y = segment["y"]
    b = segment["boundaries"]

    regions = [
        (b["x_start"], b["S1_1_start"], INTERVAL_FILL, "Diastole-like"),
        (b["S1_1_start"], b["S1_1_end"], EVENT_FILL, "P1"),
        (b["S1_1_end"], b["S2_1_start"], INTERVAL_FILL, "Systole-like"),
        (b["S2_1_start"], b["S2_1_end"], EVENT_FILL, "P2"),
        (b["S2_1_end"], b["S1_2_start"], INTERVAL_FILL, "Diastole-like"),
        (b["S1_2_start"], b["S1_2_end"], EVENT_FILL, "P1"),
        (b["S1_2_end"], b["S2_2_start"], INTERVAL_FILL, "Systole-like"),
        (b["S2_2_start"], b["S2_2_end"], EVENT_FILL, "P2"),
        (b["S2_2_end"], b["x_end"], INTERVAL_FILL, "Diastole-like"),
    ]

    for x0, x1, colour, label in regions:
        x0 = max(b["x_start"], x0)
        x1 = min(b["x_end"], x1)
        if x1 <= x0:
            continue

        centre = 0.5 * (x0 + x1)
        ax.axvspan(x0, x1, color=colour, alpha=0.92, linewidth=0, zorder=0)

        if label in ("P1", "P2"):
            ax.annotate(
                label,
                xy=(centre, 1.01),
                xytext=(centre, 1.12),
                ha="center",
                va="bottom",
                fontsize=10,
                color=DARK_TEXT,
                annotation_clip=False,
                arrowprops={
                    "arrowstyle": "-",
                    "color": SECONDARY_COLOR,
                    "linewidth": 0.9,
                    "shrinkA": 0,
                    "shrinkB": 2,
                },
            )
        else:
            edge_interval = x0 <= b["x_start"] + 1e-9 or x1 >= b["x_end"] - 1e-9
            display_label = (
                "Diastole-\nlike" if label == "Diastole-like" and edge_interval else label
            )
            ax.text(
                centre,
                0.89,
                display_label,
                ha="center",
                va="center",
                fontsize=9.5,
                color=DARK_TEXT,
                linespacing=0.9,
                zorder=5,
            )

    for boundary in [
        b["S1_1_start"],
        b["S1_1_end"],
        b["S2_1_start"],
        b["S2_1_end"],
        b["S1_2_start"],
        b["S1_2_end"],
        b["S2_2_start"],
        b["S2_2_end"],
    ]:
        if b["x_start"] <= boundary <= b["x_end"]:
            ax.axvline(
                boundary,
                color=SECONDARY_COLOR,
                linestyle=(0, (5, 3)),
                linewidth=0.9,
                alpha=0.85,
                zorder=2,
            )

    ax.plot(x, y, color=PRIMARY_COLOR, linewidth=1.7, zorder=3)

    annotation = (
        f"Autocorrelation period = {segment['selected_lag_s']:.2f} s\n"
        f"Estimated rate = {segment['estimated_rate']:.1f} bpm\n"
        f"Autocorrelation peak = {segment['autocorrelation_peak']:.2f}"
    )

    ax.text(
        0.025,
        0.045,
        annotation,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=8.7,
        color=DARK_TEXT,
        bbox={
            "boxstyle": "round,pad=0.32",
            "facecolor": "white",
            "edgecolor": GRID_COLOR,
            "linewidth": 0.8,
            "alpha": 0.94,
        },
        zorder=7,
    )

    ax.set_title(title, loc="left", fontsize=11.5, fontweight="semibold", pad=7)
    ax.set_xlabel("Time (s)")
    ax.set_xlim(b["x_start"], b["x_end"])
    ax.set_ylim(-1.05, 1.20)
    ax.margins(x=0)
    finish_axis(ax, grid_axis=None)


mic_normal = recording_results.loc[
    (recording_results["sensor_type"] == "MIC")
    & (recording_results["condition"] == "normalbreathing")
].copy()
mic_median = float(mic_normal["estimated_heart_rate_bpm"].median())
mic_normal["distance_from_median"] = (
    mic_normal["estimated_heart_rate_bpm"] - mic_median
).abs()

own_row = mic_normal.sort_values(
    ["distance_from_median", "autocorrelation_peak"], ascending=[True, False]
).iloc[0]
own_recording = recordings[own_row["recording_id"]]
own_rate = float(own_row["estimated_heart_rate_bpm"])

physionet_table = pd.DataFrame(
    [
        {
            "candidate_index": index,
            "record_name": candidate["record_name"],
            "subject_id": candidate["subject_id"],
            "estimated_heart_rate_bpm": candidate["result"]["estimated_heart_rate_bpm"],
            "autocorrelation_peak": candidate["result"]["autocorrelation_peak"],
        }
        for index, candidate in enumerate(physionet_candidates)
    ]
)
physionet_table["rate_difference"] = (
    physionet_table["estimated_heart_rate_bpm"] - own_rate
).abs()

close_matches = physionet_table.loc[physionet_table["rate_difference"] <= 10.0]
if close_matches.empty:
    close_matches = physionet_table.nsmallest(10, "rate_difference")

pn_row = close_matches.sort_values(
    ["autocorrelation_peak", "rate_difference"], ascending=[False, True]
).iloc[0]
pn_recording = physionet_candidates[int(pn_row["candidate_index"])]

own_cycle = find_autocorrelation_anchored_cycles(
    own_recording["result"], own_recording["fs_hz"]
)
pn_cycle = find_autocorrelation_anchored_cycles(pn_recording["result"], pn_recording["fs_hz"])

own_duration = required_duration(own_recording["time"], own_cycle)
pn_duration = required_duration(pn_recording["time"], pn_cycle)
common_duration = max(own_duration, pn_duration)

own_segment = build_segment(
    time_values=own_recording["time"],
    filtered_signal=own_recording["result"]["filtered_signal"],
    cycle=own_cycle,
    target_duration=common_duration,
    selected_lag_s=own_recording["result"]["selected_lag_s"],
    estimated_rate=own_recording["result"]["estimated_heart_rate_bpm"],
    autocorrelation_peak=own_recording["result"]["autocorrelation_peak"],
)

pn_segment = build_segment(
    time_values=pn_recording["time"],
    filtered_signal=pn_recording["result"]["filtered_signal"],
    cycle=pn_cycle,
    target_duration=common_duration,
    selected_lag_s=pn_recording["result"]["selected_lag_s"],
    estimated_rate=pn_recording["result"]["estimated_heart_rate_bpm"],
    autocorrelation_peak=pn_recording["result"]["autocorrelation_peak"],
)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.7), sharex=True, sharey=True)

plot_segment(
    axes[0],
    own_segment,
    "(a) Representative NilocasPatch microphone recording\nduring normal breathing",
)
plot_segment(
    axes[1],
    pn_segment,
    "(b) Representative PhysioNet recording\nfrom a subject classified as Normal",
)

axes[0].set_ylabel("Normalised amplitude")
for ax in axes:
    ax.set_xlim(0, common_duration)
    ax.set_ylim(-1.05, 1.20)

fig.subplots_adjust(left=0.075, right=0.985, bottom=0.16, top=0.82, wspace=0.16)

save_figure(fig, "MIC_vs_PhysioNet_autocorrelation_cycle_structure")
plt.show()

print("Selected NilocasPatch recording:", own_row["recording_id"])
print("Selected PhysioNet record:", pn_recording["record_name"])

### 3.3.4 Spectral signal-quality

In [ ]:
def load_labchart_txt(path):
    frame = pd.read_csv(path, sep="\t", header=None, decimal=",", engine="python")
    frame = frame.apply(pd.to_numeric, errors="coerce")

    time = frame.iloc[:, 0].to_numpy(dtype=float)
    channels = frame.iloc[:, 1:].copy()
    channels.columns = [f"ch{index+1}" for index in range(channels.shape[1])]

    return time, channels


def interpolated(values):
    return pd.Series(values).interpolate(limit_direction="both").to_numpy(dtype=float)


def representative_mean_channel(channels):
    usable_channels = []

    for column in channels.columns:
        values = channels[column].to_numpy(dtype=float)
        if np.mean(np.isnan(values)) <= 0.30 and np.nanstd(values) > 0:
            usable_channels.append(column)

    if not usable_channels:
        raise ValueError("No usable channels in recording.")

    mean_signal = np.nanmean(channels[usable_channels].to_numpy(dtype=float), axis=1)
    return mean_signal, usable_channels


def bandpass_signal(values, fs):
    values = interpolated(values)
    sos = signal.butter(4, [15, 90], btype="bandpass", fs=fs, output="sos")
    return signal.sosfiltfilt(sos, values)


def spectral_power_ratio_db(values, fs):
    values = interpolated(values)
    values = values - np.mean(values)

    nperseg = min(len(values), int(round(8 * fs)))
    noverlap = nperseg // 2

    frequencies, psd = signal.welch(
        values, fs=fs, window="hann", nperseg=nperseg, noverlap=noverlap, detrend=False
    )

    cardiac_mask = (frequencies >= 15) & (frequencies <= 90)
    reference_mask = (frequencies >= 200) & (frequencies <= 500)

    cardiac_power = np.trapezoid(psd[cardiac_mask], frequencies[cardiac_mask])
    reference_power = np.trapezoid(psd[reference_mask], frequencies[reference_mask])

    return 10 * np.log10(cardiac_power / reference_power)


processed_recordings = {}
quality_rows = []

for row in volunteer_manifest.itertuples(index=False):
    time, channels = load_labchart_txt(row.filepath)

    sampling_interval = np.nanmedian(np.diff(time))
    fs = 1.0 / sampling_interval

    mean_signal, usable_channels = representative_mean_channel(channels)
    filtered_signal = bandpass_signal(mean_signal, fs)

    recording_id = f"{row.sensor_type}_subj{row.subject_id}_{row.condition}"

    processed_recordings[recording_id] = {
        "time": time,
        "fs": fs,
        "mean_signal": mean_signal,
        "filtered_signal": filtered_signal,
        "usable_channels": usable_channels,
        "sensor_type": row.sensor_type,
        "subject_id": row.subject_id,
        "condition": row.condition,
        "filename": row.filename,
    }

    quality_rows.append(
        {
            "recording_id": recording_id,
            "subject_id": row.subject_id,
            "sensor_type": row.sensor_type,
            "condition": row.condition,
            "duration_s": time[-1] - time[0],
            "fs_estimated_hz": fs,
            "n_usable_channels": len(usable_channels),
            "snr_psd_based_db": spectral_power_ratio_db(mean_signal, fs),
        }
    )

quality_summary = pd.DataFrame(quality_rows)
display(quality_summary)

#### Figure 19 — PSD-based spectral power ratio across sensing modalities and breathing conditions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6), sharey=True)

for ax, condition, title in [
    (axes[0], "normal", "(a) ACC and MIC during normal breathing"),
    (axes[1], "hold", "(b) ACC and MIC during breath hold"),
]:
    subset = quality_summary.loc[quality_summary["condition"] == condition].copy()

    acc_values = (
        subset.loc[subset["sensor_type"] == "ACC"]
        .set_index("subject_id")["snr_psd_based_db"]
        .sort_index()
    )
    mic_values = (
        subset.loc[subset["sensor_type"] == "MIC"]
        .set_index("subject_id")["snr_psd_based_db"]
        .sort_index()
    )

    common_subjects = acc_values.index.intersection(mic_values.index)

    boxplot = ax.boxplot(
        [acc_values.loc[common_subjects], mic_values.loc[common_subjects]],
        positions=[1, 2],
        widths=0.44,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": DARK_TEXT, "linewidth": 1.8},
        whiskerprops={"linewidth": 1.2},
        capprops={"linewidth": 1.2},
        boxprops={"linewidth": 0},
    )

    boxplot["boxes"][0].set_facecolor(SECONDARY_COLOR)
    boxplot["boxes"][0].set_alpha(0.55)
    boxplot["boxes"][1].set_facecolor(PRIMARY_COLOR)
    boxplot["boxes"][1].set_alpha(0.55)

    for subject_id in common_subjects:
        ax.plot(
            [1, 2],
            [acc_values.loc[subject_id], mic_values.loc[subject_id]],
            color=GRID_COLOR,
            linewidth=1.2,
            alpha=0.9,
            zorder=1,
        )

    ax.scatter(
        np.full(len(common_subjects), 1),
        acc_values.loc[common_subjects],
        color=SECONDARY_COLOR,
        s=38,
        linewidths=0,
        zorder=3,
    )
    ax.scatter(
        np.full(len(common_subjects), 2),
        mic_values.loc[common_subjects],
        color=PRIMARY_COLOR,
        s=38,
        linewidths=0,
        zorder=3,
    )

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["ACC", "MIC"])
    ax.set_title(title, loc="left", fontsize=13, fontweight="semibold", pad=7)
    ax.set_ylabel("PSD-based spectral power ratio (dB)")
    finish_axis(ax)

fig.tight_layout()
save_figure(fig, "healthy_volunteers_psd_paired_boxplots")
for ax in axes:
    ax.set_ylim(-5, 32)
    ax.set_yticks(np.arange(-5, 31, 5))
plt.show()

### 3.3.5 Device comfort

#### Figure 20 — Device comfort, tolerability, usability and acceptability

In [ ]:
responses = pd.DataFrame(
    {
        "Subject": [
            "Subject 1",
            "Subject 2",
            "Subject 3",
            "Subject 4",
            "Subject 5",
            "Subject 6",
        ],
        "Comfort": [
            "Comfortable",
            "Comfortable",
            "Very comfortable",
            "Very uncomfortable",
            "Very uncomfortable",
            "Neutral",
        ],
        "Discomfort": ["No discomfort"] * 6,
        "Weight": [
            "Lightweight",
            "Lightweight",
            "Just right",
            "Just right",
            "Lightweight",
            "Just right",
        ],
        "Skin reaction": ["No"] * 6,
        "Application": ["Neutral", "Very easy", "Neutral", "Neutral", "Neutral", "Neutral"],
        "Fit": ["Slightly too loose", "Secure", "Secure", "Secure", "Secure", "Secure"],
        "Movement restriction": [
            "Slightly",
            "Slightly",
            "Not at all",
            "Not at all",
            "Not at all",
            "Slightly",
        ],
        "Willingness": ["Yes", "Yes", "Yes", "Unsure", "Yes", "Unsure"],
        "Satisfaction": [
            "Satisfied",
            "Neutral",
            "Very satisfied",
            "Satisfied",
            "Very satisfied",
            "Neutral",
        ],
    }
)

n_subjects = len(responses)

assessment = pd.DataFrame(
    {
        "Measure": [
            "Comfortable or\nvery comfortable",
            "No discomfort",
            "Acceptable weight",
            "No skin reaction",
            "Easy or very easy\nto apply/remove",
            "Secure fit",
            "Minimal movement\nrestriction",
            "Willing to use\nregularly",
            "Satisfied or\nvery satisfied",
        ],
        "Domain": [
            "Safety and tolerability",
            "Safety and tolerability",
            "Safety and tolerability",
            "Safety and tolerability",
            "Usability and acceptability",
            "Usability and acceptability",
            "Usability and acceptability",
            "Usability and acceptability",
            "Usability and acceptability",
        ],
        "Positive responses": [
            responses["Comfort"].isin(["Comfortable", "Very comfortable"]).sum(),
            responses["Discomfort"].eq("No discomfort").sum(),
            responses["Weight"].isin(["Just right", "Lightweight"]).sum(),
            responses["Skin reaction"].eq("No").sum(),
            responses["Application"].isin(["Easy", "Very easy"]).sum(),
            responses["Fit"].eq("Secure").sum(),
            responses["Movement restriction"].isin(["Not at all", "Slightly"]).sum(),
            responses["Willingness"].eq("Yes").sum(),
            responses["Satisfaction"].isin(["Satisfied", "Very satisfied"]).sum(),
        ],
    }
)

assessment["Percentage"] = 100 * assessment["Positive responses"] / n_subjects


def questionnaire_panel(ax, data, title, color):
    plot_data = data.iloc[::-1].reset_index(drop=True)
    positions = np.arange(len(plot_data))

    ax.barh(
        positions, plot_data["Percentage"], height=0.38, color=color, alpha=0.88, linewidth=0
    )

    ax.set_yticks(positions)
    ax.set_yticklabels(plot_data["Measure"])

    for y_position, row in plot_data.iterrows():
        percentage = float(row["Percentage"])
        count = int(row["Positive responses"])
        label = f"{percentage:.0f}%"

        x_position = percentage - 2 if percentage >= 35 else max(percentage, 1)
        ax.text(
            x_position,
            y_position,
            label,
            ha="right",
            va="center",
            color="white",
            fontsize=10.5,
            fontweight="semibold",
        )

    ax.set_title(title, loc="left", fontsize=13, fontweight="semibold", pad=7)
    ax.set_xlabel("Subjects reporting a positive response (%)")
    ax.set_xlim(0, 105)
    ax.xaxis.set_major_locator(MultipleLocator(20))
    finish_axis(ax, grid_axis="x")


safety = assessment.loc[assessment["Domain"] == "Safety and tolerability"]
usability = assessment.loc[assessment["Domain"] == "Usability and acceptability"]

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6), sharex=True)

questionnaire_panel(axes[0], safety, "(a) Safety and tolerability", PRIMARY_COLOR)
questionnaire_panel(axes[1], usability, "(b) Usability and acceptability", SECONDARY_COLOR)

fig.subplots_adjust(wspace=0.48, top=0.86, bottom=0.18, left=0.18, right=0.97)

save_figure(fig, "device_comfort_usability_acceptability")
plt.show()

display(assessment)